# Student Mental Health Prediction — Deployment

## Objectif

Après avoir entraîné, évalué et optimisé plusieurs modèles de Machine Learning, cette étape consiste à rendre le modèle final utilisable à travers une application.

La Logistic Regression optimisée a été sélectionnée comme modèle final sur la base de ses performances obtenues sur le jeu de test.

L'objectif du déploiement est de permettre à un utilisateur de fournir les caractéristiques d'un étudiant et d'obtenir une prédiction à partir du modèle entraîné.

## 1 Chargement du jeu de test

Le jeu de test préparé lors de la phase de Data Preparation est chargé afin de réaliser une première vérification du modèle sauvegardé.

Cette étape permet de vérifier que le modèle chargé peut recevoir les mêmes variables que celles utilisées lors de son entraînement.

In [2]:
import pandas as pd

X_test = pd.read_csv("../dataset/processed/X_test.csv")

print("Dimensions de X_test :", X_test.shape)
print("Nombre de variables :", X_test.shape[1])

Dimensions de X_test : (5580, 15)
Nombre de variables : 15


In [3]:
print(X_test.head())

   Gender       Age  Profession  Academic Pressure  Work Pressure      CGPA  \
0       0 -0.174916          12          -0.829360      -0.008701  0.624006   
1       1 -0.988582          12          -0.829360      -0.008701 -1.249707   
2       0  1.655833          12          -1.555513      -0.008701  0.114845   
3       0  1.452417          12          -0.103206      -0.008701  1.540496   
4       1 -1.598832          12           1.349100      -0.008701 -0.435049   

   Study Satisfaction  Job Satisfaction  Sleep Duration  Dietary Habits  \
0            0.039692          -0.01608               1               0   
1           -1.428155          -0.01608               0               1   
2            1.507539          -0.01608               0               2   
3            1.507539          -0.01608               0               1   
4           -1.428155          -0.01608               0               0   

   Degree  Have you ever had suicidal thoughts ?  Work/Study Hours  \
0   

In [4]:
print("Dimensions de X_test :", X_test.shape)
print("Nombre de variables :", X_test.shape[1])

Dimensions de X_test : (5580, 15)
Nombre de variables : 15


## 2. Chargement du modèle final

Le modèle sélectionné lors de la phase de modélisation est la Logistic Regression optimisée.

Le modèle sauvegardé au format `.pkl` sera chargé afin d'être réutilisé pour effectuer des prédictions sans devoir réentraîner le modèle.

In [5]:
import joblib
import os

In [6]:
model_path = "../models/student_depression_model.pkl"

model = joblib.load(model_path)

print("Modèle chargé avec succès.")
print(model)

Modèle chargé avec succès.
LogisticRegression(C=0.01, max_iter=1000, random_state=42)


## 3. Vérification du modèle

Une vérification est réalisée afin de confirmer que le modèle chargé correspond bien au modèle final sélectionné lors de la phase de modélisation.

In [7]:
print("Type du modèle :", type(model))
print("Nombre de variables attendues :", model.n_features_in_)

Type du modèle : <class 'sklearn.linear_model._logistic.LogisticRegression'>
Nombre de variables attendues : 15


## 4. Test d'une prédiction

Avant de construire l'interface utilisateur, une première prédiction est réalisée à partir d'une observation provenant du jeu de test.

Cette étape permet de vérifier que le modèle chargé peut être utilisé correctement pour effectuer une prédiction.

In [8]:
sample = X_test.iloc[[0]]

print("Dimensions de l'observation :", sample.shape)

Dimensions de l'observation : (1, 15)


In [9]:
prediction = model.predict(sample)

print("Prédiction :", prediction[0])

Prédiction : 1


In [10]:
probabilities = model.predict_proba(sample)[0]

print("Probabilité classe 0 :", probabilities[0])
print("Probabilité classe 1 :", probabilities[1])

Probabilité classe 0 : 0.06773608749856486
Probabilité classe 1 : 0.9322639125014351


## 5. Création d'une fonction de prédiction

Afin de faciliter la réutilisation du modèle dans l'application, une fonction de prédiction est créée.

Cette fonction reçoit une observation préparée, utilise le modèle sauvegardé pour effectuer la prédiction et retourne à la fois la classe prédite et les probabilités associées.

In [11]:
def predict_student(data):
    prediction = model.predict(data)[0]
    probabilities = model.predict_proba(data)[0]

    return prediction, probabilities

### Test de la fonction de prédiction

La fonction est testée sur l'observation utilisée précédemment afin de vérifier qu'elle retourne correctement la prédiction et les probabilités associées.

In [12]:
prediction, probabilities = predict_student(sample)

print("Prédiction :", prediction)
print("Probabilité classe 0 :", probabilities[0])
print("Probabilité classe 1 :", probabilities[1])

Prédiction : 1
Probabilité classe 0 : 0.06773608749856486
Probabilité classe 1 : 0.9322639125014351


## 6. Préparation des données pour le déploiement

Dans le notebook de modélisation, le modèle a été entraîné à partir de données déjà préparées et transformées.

Lors du déploiement, l'utilisateur fournira cependant des informations sous une forme compréhensible, par exemple `Female`, `Yes`, `Healthy`, etc.

Il est donc nécessaire de reproduire les transformations appliquées lors du Data Preparation afin de convertir les données utilisateur dans le format attendu par le modèle.

Cette étape permet d'assurer la cohérence entre les données utilisées lors de l'entraînement et les nouvelles données reçues par l'application.

### 6.1 Identification des variables d'entrée

Le modèle final a été entraîné avec 15 variables explicatives.

Avant de construire l'interface utilisateur, il est nécessaire d'identifier précisément ces variables ainsi que leur ordre. L'application devra fournir les données dans le même ordre que celui utilisé lors de l'entraînement du modèle.

In [13]:
print("Variables utilisées par le modèle :")
print(X_test.columns.tolist())

Variables utilisées par le modèle :
['Gender', 'Age', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness']


In [14]:
print("Nombre de variables :", len(X_test.columns))

Nombre de variables : 15


### 6.2 Identification des variables catégorielles et numériques

Les variables utilisées par le modèle peuvent être de nature numérique ou catégorielle.

Cette distinction est importante pour le déploiement, car les réponses saisies par l'utilisateur devront être transformées dans le même format que celui utilisé lors de l'entraînement.

In [15]:
print(X_test.dtypes)

Gender                                     int64
Age                                      float64
Profession                                 int64
Academic Pressure                        float64
Work Pressure                            float64
CGPA                                     float64
Study Satisfaction                       float64
Job Satisfaction                         float64
Sleep Duration                             int64
Dietary Habits                             int64
Degree                                     int64
Have you ever had suicidal thoughts ?      int64
Work/Study Hours                         float64
Financial Stress                         float64
Family History of Mental Illness           int64
dtype: object


In [16]:
sample = X_test.iloc[[0]]

print(sample.T)

                                               0
Gender                                  0.000000
Age                                    -0.174916
Profession                             12.000000
Academic Pressure                      -0.829360
Work Pressure                          -0.008701
CGPA                                    0.624006
Study Satisfaction                      0.039692
Job Satisfaction                       -0.016080
Sleep Duration                          1.000000
Dietary Habits                          0.000000
Degree                                 21.000000
Have you ever had suicidal thoughts ?   1.000000
Work/Study Hours                        1.034347
Financial Stress                        1.295662
Family History of Mental Illness        1.000000


### 6.3 Préparation d'une observation pour la prédiction

Afin de tester le fonctionnement du modèle dans des conditions proches du futur déploiement, une observation est construite à partir des valeurs d'entrée attendues par le modèle.

L'observation doit respecter le même nombre de variables et le même ordre que les données utilisées lors de l'entraînement.

In [17]:
feature_names = X_test.columns.tolist()

print("Variables attendues par le modèle :")
for i, feature in enumerate(feature_names, start=1):
    print(i, "→", feature)

Variables attendues par le modèle :
1 → Gender
2 → Age
3 → Profession
4 → Academic Pressure
5 → Work Pressure
6 → CGPA
7 → Study Satisfaction
8 → Job Satisfaction
9 → Sleep Duration
10 → Dietary Habits
11 → Degree
12 → Have you ever had suicidal thoughts ?
13 → Work/Study Hours
14 → Financial Stress
15 → Family History of Mental Illness


In [18]:
sample = X_test.iloc[[0]].copy()

print(sample)

   Gender       Age  Profession  Academic Pressure  Work Pressure      CGPA  \
0       0 -0.174916          12           -0.82936      -0.008701  0.624006   

   Study Satisfaction  Job Satisfaction  Sleep Duration  Dietary Habits  \
0            0.039692          -0.01608               1               0   

   Degree  Have you ever had suicidal thoughts ?  Work/Study Hours  \
0      21                                      1          1.034347   

   Financial Stress  Family History of Mental Illness  
0          1.295662                                 1  


In [19]:
prediction, probabilities = predict_student(sample)

print("Prédiction :", prediction)
print("Probabilité classe 0 :", probabilities[0])
print("Probabilité classe 1 :", probabilities[1])

Prédiction : 1
Probabilité classe 0 : 0.06773608749856486
Probabilité classe 1 : 0.9322639125014351


# 7. Développement de l'application Flask

Flask est un framework Python léger permettant de créer des applications web.

Dans ce projet, Flask sera utilisé comme intermédiaire entre l'interface utilisateur et le modèle de Machine Learning.

L'application permettra à l'utilisateur de renseigner les caractéristiques d'un étudiant, puis transmettra ces informations au modèle afin d'obtenir une prédiction.

In [20]:
! pip install flask


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import flask; print(flask.__version__)

3.1.3


C:\Users\Hp\AppData\Local\Temp\ipykernel_6236\1295262177.py:1: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  import flask; print(flask.__version__)


# 9. Création de l'interface utilisateur

Une application web a besoin d'une interface permettant à l'utilisateur d'interagir avec le modèle.

HTML (HyperText Markup Language) sera utilisé pour créer cette interface.

Dans notre projet, la page HTML contiendra un formulaire permettant à l'utilisateur de renseigner les caractéristiques nécessaires à la prédiction.

Le fonctionnement sera le suivant :

Utilisateur → Formulaire HTML → Flask → Modèle → Prédiction

## 10. Création du formulaire de prédiction

L'étape suivante consiste à créer une interface permettant à l'utilisateur de saisir les caractéristiques d'un étudiant.

Ces informations seront ensuite envoyées à l'application Flask.

Flask récupérera les valeurs saisies, les préparera dans le même format que les données utilisées lors de l'entraînement du modèle, puis les transmettra au modèle de Machine Learning afin d'obtenir une prédiction.

Le processus sera donc :

Utilisateur
↓
Formulaire HTML
↓
Flask
↓
Préparation des données
↓
Modèle de Machine Learning
↓
Prédiction
↓
Résultat affiché à l'utilisateur

In [22]:
model.feature_names_in_

array(['Gender', 'Age', 'Profession', 'Academic Pressure',
       'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction',
       'Sleep Duration', 'Dietary Habits', 'Degree',
       'Have you ever had suicidal thoughts ?', 'Work/Study Hours',
       'Financial Stress', 'Family History of Mental Illness'],
      dtype=object)

In [23]:
X_train = pd.read_csv("../dataset/processed/X_train.csv")

print("Dimensions de X_train :", X_train.shape)

Dimensions de X_train : (22318, 15)


In [24]:
X_train.columns.to_list()

['Gender',
 'Age',
 'Profession',
 'Academic Pressure',
 'Work Pressure',
 'CGPA',
 'Study Satisfaction',
 'Job Satisfaction',
 'Sleep Duration',
 'Dietary Habits',
 'Degree',
 'Have you ever had suicidal thoughts ?',
 'Work/Study Hours',
 'Financial Stress',
 'Family History of Mental Illness']

In [25]:
X_train.head()

,Gender,Age,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness
0,1,-1.598832,12,-0.103206,-0.008701,1.560863,1.507539,-0.01608,2,2,0,1,-1.396430,0.599648,1
1,0,-1.395415,12,1.349100,-0.008701,-1.141086,-0.694231,-0.01608,0,2,0,1,0.764260,1.295662,1
2,0,-0.378332,12,0.622947,-0.008701,-0.625136,0.773615,-0.01608,0,0,11,1,0.224088,1.295662,0
3,1,0.435334,12,-0.103206,-0.008701,1.486186,0.773615,-0.01608,0,1,8,1,0.764260,1.295662,1
4,0,-1.598832,12,-0.829360,-0.008701,1.173900,1.507539,-0.01608,1,2,0,1,1.304433,0.599648,0


### 10.3 Exploration des valeurs des variables

Avant de construire le formulaire HTML, les valeurs possibles de chaque variable catégorielle sont examinées.

Cette étape permet de construire des champs de formulaire cohérents avec les données utilisées lors de l'entraînement du modèle.

Les catégories proposées à l'utilisateur doivent correspondre aux valeurs présentes dans les données d'entraînement.

In [26]:
categorical_columns = [
    "Gender",
    "Profession",
    "Sleep Duration",
    "Dietary Habits",
    "Degree",
    "Have you ever had suicidal thoughts ?",
    "Family History of Mental Illness"
]

for column in categorical_columns:
    print("\n", column)
    print(X_train[column].unique())


 Gender
[1 0]

 Profession
[12  0  1  5  6 11  7 13  2  9 10  3]

 Sleep Duration
[2 0 1 3 4]

 Dietary Habits
[2 0 1 3]

 Degree
[ 0 11  8 15 25 22  3 18 21 16 14  1 10  9  4  5 13 20  2 17 24  7 19 12
 27 23  6 26]

 Have you ever had suicidal thoughts ?
[1 0]

 Family History of Mental Illness
[1 0]


In [27]:
numerical_columns = [
    "Age",
    "Academic Pressure",
    "Work Pressure",
    "CGPA",
    "Study Satisfaction",
    "Job Satisfaction",
    "Work/Study Hours",
    "Financial Stress"
]

for column in numerical_columns:
    print("\n", column)
    print("Min :", X_train[column].min())
    print("Max :", X_train[column].max())
    print("Valeurs uniques :", X_train[column].unique()[:20])


 Age
Min : -1.5988315640687285
Max : 6.741247428972543
Valeurs uniques : [-1.59883156 -1.395415   -0.3783322   0.43533404 -1.19199844  0.6387506
 -0.98858188 -0.58174876  1.45241685  1.24900029  0.84216717 -0.17491564
  1.65583341  0.02850092 -0.78516532  1.04558373  0.23191748  3.48658246
  6.53783087  3.2831659 ]

 Academic Pressure
Min : -2.2816659744566574
Max : 1.3491001620635836
Valeurs uniques : [-0.10320629  1.34910016  0.62294693 -0.82935952 -1.55551275 -2.28166597]

 Work Pressure
Min : -0.0087013741824627
Max : 138.70363362882
Valeurs uniques : [-8.70137418e-03  5.54762326e+01  1.38703634e+02]

 CGPA
Min : -5.200797273644287
Max : 1.5880181689440516
Valeurs uniques : [ 1.56086291 -1.14108564 -0.62513567  1.48618594  1.17390043 -1.76565666
 -1.11393038 -0.45541528 -0.12276332 -1.75207903 -0.38073831  0.94986952
 -1.62309154  1.39793134  0.34566494  0.69868335 -1.30401721 -0.76770079
  1.50655238  0.25741034]

 Study Satisfaction
Min : -2.162077834780113
Max : 1.5075385063930

### 10.4 Compréhension du preprocessing utilisé par le modèle

L'analyse de `X_train` montre que les variables utilisées par le modèle ont déjà subi des transformations.

Les variables numériques, telles que `Age`, `Academic Pressure`, `CGPA` et `Financial Stress`, sont représentées sous une échelle transformée.

Les variables catégorielles sont également représentées sous forme numérique.

Par conséquent, l'application de déploiement ne doit pas envoyer directement les valeurs brutes saisies par l'utilisateur au modèle.

Les mêmes transformations utilisées lors de la préparation des données doivent être appliquées aux nouvelles observations avant la prédiction.

In [28]:
import joblib

model = joblib.load("../models/student_depression_model.pkl")

print("Modèle chargé avec succès.")
print(type(model))

Modèle chargé avec succès.
<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [29]:
print("Nombre de variables :", len(model.feature_names_in_))
print("Variables utilisées :")
print(model.feature_names_in_)

Nombre de variables : 15
Variables utilisées :
['Gender' 'Age' 'Profession' 'Academic Pressure' 'Work Pressure' 'CGPA'
 'Study Satisfaction' 'Job Satisfaction' 'Sleep Duration' 'Dietary Habits'
 'Degree' 'Have you ever had suicidal thoughts ?' 'Work/Study Hours'
 'Financial Stress' 'Family History of Mental Illness']


## 12. Préparation des données provenant du formulaire

L'application web recevra les informations saisies par l'utilisateur sous forme de valeurs provenant d'un formulaire HTML.

Ces valeurs devront être organisées selon les 15 variables utilisées par le modèle final.

Avant de réaliser la prédiction, les données reçues devront être converties dans le même format que celui utilisé lors de l'entraînement du modèle.

Cette étape permet d'assurer la compatibilité entre les données provenant de l'application web et le modèle de Machine Learning.